<a href="https://colab.research.google.com/github/suyaibalsifat/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suyaibalsifat/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!git clone https://github.com/SUYAIBALSIFAT/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 142, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 142 (delta 53), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (142/142), 1.86 MiB | 10.40 MiB/s, done.
Resolving deltas: 100% (53/53), done.
/content/flyrank-ml-internship


## 1. My rule and its reason codes

Signal check 1: staleness (days_since_last_update >= 180) vs decline rate.
Result: stale pages decline LESS (47.1%, n=174) than fresh pages (54.2%, n=29,826).
Verdict: MIXED — direction is opposite of what the refresh-flag logic assumes, but the stale group is tiny (n=174 vs 29,826), so I don't trust this enough to call it a clean OPPOSITE. I'm dropping staleness from my rule; a small, weird sample isn't a base to build a score on.

Signal check 2: low CTR (ctr < 0.5) on visible pages (impressions_90d >= 500, position 1-20) vs decline rate.
Result: low-CTR pages decline MORE (62.7%, n=9,759) than higher-CTR pages (47.5%, n=2,264).
Verdict: CONFIRMED — this is a real, well-sampled signal, and it lines up with FlyRank's own CTR-fix logic from the session.

My rule (plain words): flag a page if it's (a) visibly declining with real search demand, or (b) visible but under-capturing clicks for its position. Score it higher the more impressions it has, since fixing a high-traffic page matters more than a low-traffic one.

Reason codes: declining_and_low_ctr, declining_with_demand, ctr_review_candidate.
Action labels: refresh_review, ctr_review.

In [ ]:
# No code

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import pandas as pd
import os

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

declining = (df.trend_direction == "down") & (df.impressions_90d >= 100)
low_ctr = (df.impressions_90d >= 500) & (df.avg_position.between(0, 20, inclusive="right")) & (df.ctr < 0.5)

df["declining_flag"] = declining.astype(int)
df["low_ctr_flag"] = low_ctr.astype(int)

# ONE reason code per row, most specific first
def reason_code(row):
    if row.declining_flag and row.low_ctr_flag:
        return "declining_and_low_ctr"
    elif row.declining_flag:
        return "declining_with_demand"
    elif row.low_ctr_flag:
        return "ctr_review_candidate"
    return "no_flag"

df["reason_code"] = df.apply(reason_code, axis=1)
df["action"] = df["reason_code"].map({
    "declining_and_low_ctr": "refresh_review",
    "declining_with_demand": "refresh_review",
    "ctr_review_candidate": "ctr_review",
    "no_flag": "monitor",
})

# score: weight by impressions so high-traffic flagged pages rank first
df["baseline_score"] = df["impressions_90d"] * (df["declining_flag"] + df["low_ctr_flag"])

queue = df[df.reason_code != "no_flag"].sort_values("baseline_score", ascending=False)
print(f"Flagged pages: {len(queue)} of {len(df)}")

os.makedirs("work/outputs", exist_ok=True)
out_cols = ["content_id", "baseline_score", "reason_code", "action",
            "impressions_90d", "trend_direction", "avg_position", "ctr"]
queue[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
queue[out_cols].head(10)


Flagged pages: 16791 of 30000


,content_id,baseline_score,reason_code,action,impressions_90d,trend_direction,avg_position,ctr
6653,content_5fe46e04994d,1035430,declining_and_low_ctr,refresh_review,517715,down,4.2,0.14
26844,content_8c19996aa890,1018504,declining_and_low_ctr,refresh_review,509252,down,2.5,0.15
21819,content_4c36c775b818,926206,declining_and_low_ctr,refresh_review,463103,down,2.3,0.41
29879,content_1a9e894be2e2,832360,declining_and_low_ctr,refresh_review,416180,down,4.0,0.23
26531,content_cb112fce36be,619820,declining_and_low_ctr,refresh_review,309910,down,5.6,0.16
17812,content_aaef01a50def,517109,ctr_review_candidate,ctr_review,517109,stable,5.4,0.25
27478,content_008fb02c46cb,473606,declining_and_low_ctr,refresh_review,236803,down,4.4,0.26
11655,content_cea79ef51519,417596,declining_and_low_ctr,refresh_review,208798,down,5.2,0.23
7445,content_c8e9d6ab9013,417356,declining_and_low_ctr,refresh_review,208678,down,9.7,0.00
16950,content_bf7bff5d0756,394398,declining_and_low_ctr,refresh_review,197199,down,6.8,0.22


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

1. content_5fe46e04994d — refresh_review, declining_and_low_ctr. Huge visibility (517,715 impressions), trending down, and CTR (0.14) far below what a position-4 page should get. Strong double signal, highest priority in the queue. Would be wrong if: a sitewide tracking change temporarily distorted CTR measurement for this period.

2. content_8c19996aa890 — refresh_review, declining_and_low_ctr. Similar profile: ~509K impressions, position 2.5, CTR only 0.15. A page ranking that well should pull much higher CTR. Would be wrong if: this page recently lost a featured snippet or rich result, which drops CTR without the content itself being worse.

3. content_4c36c775b818 — refresh_review, declining_and_low_ctr. CTR here (0.41) is much closer to my 0.5 cutoff than rows 1-2 — this is my most borderline "low CTR" call in the top 10. Would be wrong if: normal week-to-week noise pushed it just under the threshold; a slightly different cutoff might drop this one from the queue entirely.

4. content_1a9e894be2e2 — refresh_review, declining_and_low_ctr. Good position (4.0), 416K impressions, CTR 0.23 — solid double-signal case. Would be wrong if: this topic has genuine seasonal demand and the "decline" is calendar-driven rather than lasting.

5. content_cb112fce36be — refresh_review, declining_and_low_ctr. Position slipped to 5.6, CTR 0.16 on real demand (309K impressions). Would be wrong if: a nearby/related page on the same site absorbed this traffic (consolidation, not decline).

6. content_aaef01a50def — ctr_review, ctr_review_candidate. Different case from the rest: trend_direction is "stable," not "down" — this page isn't losing traffic, it's just underperforming its position (CTR 0.25 at position 5.4) even though demand is flat. Good example of the CTR-only reason code working as intended, separate from decline. Would be wrong if: this topic naturally gets lower CTR regardless of position (e.g. informational vs. transactional intent).

7. content_008fb02c46cb — refresh_review, declining_and_low_ctr. Lower impressions than the top 5 (236K) but still a real, well-positioned page (4.4) losing clicks (CTR 0.26). Would be wrong if: a competitor recently launched a stronger result at that same position, unrelated to this page's own quality.

8. content_cea79ef51519 — refresh_review, declining_and_low_ctr. Position 5.2, CTR 0.23, consistent with the rest of the pattern. Would be wrong if: the drop is very recent and still within normal week-to-week variance rather than a sustained trend.

9. content_c8e9d6ab9013 — refresh_review, declining_and_low_ctr. Notable outlier: CTR is exactly 0.00 despite 208K impressions and a decent position (9.7). This looks less like "underperforming" and more like a possible tracking/data issue — worth flagging for a sanity check rather than trusting blindly. Would be wrong if: a tracking or measurement bug is causing zero recorded clicks rather than an actual CTR problem.

10. content_bf7bff5d0756 — refresh_review, declining_and_low_ctr. Position 6.8, CTR 0.22, matches the broader pattern of visible-but-declining pages. Would be wrong if: this page was recently deindexed/reindexed and the numbers reflect a temporary gap, not ongoing decline.

In [4]:
top20 = queue[out_cols].head(20)
top20


,content_id,baseline_score,reason_code,action,impressions_90d,trend_direction,avg_position,ctr
6653,content_5fe46e04994d,1035430,declining_and_low_ctr,refresh_review,517715,down,4.2,0.14
26844,content_8c19996aa890,1018504,declining_and_low_ctr,refresh_review,509252,down,2.5,0.15
21819,content_4c36c775b818,926206,declining_and_low_ctr,refresh_review,463103,down,2.3,0.41
29879,content_1a9e894be2e2,832360,declining_and_low_ctr,refresh_review,416180,down,4.0,0.23
26531,content_cb112fce36be,619820,declining_and_low_ctr,refresh_review,309910,down,5.6,0.16
17812,content_aaef01a50def,517109,ctr_review_candidate,ctr_review,517109,stable,5.4,0.25
27478,content_008fb02c46cb,473606,declining_and_low_ctr,refresh_review,236803,down,4.4,0.26
11655,content_cea79ef51519,417596,declining_and_low_ctr,refresh_review,208798,down,5.2,0.23
7445,content_c8e9d6ab9013,417356,declining_and_low_ctr,refresh_review,208678,down,9.7,0.00
16950,content_bf7bff5d0756,394398,declining_and_low_ctr,refresh_review,197199,down,6.8,0.22


## 4. Weak picks + leakage check

Weakest pick: row 3 (content_4c36c775b818) — its CTR of 0.41 sits close to my 0.5 cutoff, so this flag is sensitive to exactly where I draw the line; a slightly stricter threshold would drop it. Row 9 (content_c8e9d6ab9013) is also worth a second look — CTR of exactly 0.00 on real impressions looks more like a possible data/tracking gap than a typical "underperforming" page, and I'd want to verify that before acting on it.

Leakage check: my score only uses impressions_90d, trend_direction, avg_position, and ctr — all things known right now, no FlyRank product decision fields (health_score, priority_score, action_type aren't in this dataset at all), and no future-window data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.